In [ ]:
### Evaluating Fine-Tuned LLM Performance
import torch
import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
save_dir = "./fine_tuned_bert_imdb"

model = AutoModelForSequenceClassification.from_pretrained(save_dir, num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [80]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(save_dir)

In [64]:
ds = load_dataset("imdb")

In [65]:
def tokenize_function(data):
    return tokenizer(data["text"], padding="max_length", truncation=True)

tokenized_datasets = ds.map(tokenize_function, batched=True)
small_test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

Map: 100%|██████████| 25000/25000 [00:10<00:00, 2428.36 examples/s]


In [ ]:
test_accuracy = tf.keras.metrics.Accuracy()
test_precision = tf.keras.metrics.Precision()
test_recall = tf.keras.metrics.Recall()
test_f1score = tf.keras.metrics.F1Score(average='macro')
ds_test_batch = small_test_dataset.batch(10)

In [156]:
print(ds_test_batch['label'])
ds_test = np.array([item for sublist in ds_test_batch['label'] for item in sublist])
print(ds_test)

[[1, 1, 0, 1, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 1, 1, 0, 0, 1, 1], [0, 0, 0, 1, 0, 1, 0, 1, 1, 1], [0, 0, 1, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [1, 1, 0, 1, 1, 1, 0, 1, 1, 1], [0, 0, 0, 0, 1, 0, 0, 1, 1, 0], [0, 0, 0, 1, 0, 0, 0, 1, 0, 1], [0, 1, 0, 1, 1, 1, 1, 0, 1, 1], [0, 1, 0, 1, 0, 0, 0, 0, 1, 1], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0], [1, 1, 1, 1, 0, 1, 1, 0, 1, 1], [0, 0, 0, 1, 1, 1, 0, 1, 0, 1], [0, 0, 0, 1, 1, 1, 0, 0, 1, 1], [0, 1, 1, 0, 1, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 1, 1, 1, 0], [0, 1, 0, 0, 0, 1, 1, 0, 0, 0], [1, 1, 0, 0, 0, 0, 1, 1, 1, 1], [1, 1, 1, 0, 0, 1, 0, 0, 1, 0], [1, 0, 0, 1, 1, 1, 0, 1, 1, 1], [1, 0, 0, 0, 0, 0, 1, 0, 0, 0], [1, 1, 1, 0, 0, 1, 1, 1, 0, 1], [0, 0, 0, 0, 1, 1, 1, 1, 1, 0], [0, 0, 0, 1, 1, 1, 1, 0, 1, 0], [1, 1, 0, 0, 1, 0, 1, 0, 1, 1], [0, 0, 0, 1, 1, 1, 1, 1, 1, 1], [1, 0, 1, 0, 0, 1, 1, 0, 1, 0], [1, 1, 0, 1, 1, 1, 0, 1, 1, 1], [1, 0, 0, 1, 0, 1, 0, 1, 1, 0], [1, 0, 0, 1, 0, 1, 1, 0, 0, 0], [1, 1, 1, 0, 1, 1, 1, 1, 0, 1], [1, 0, 

In [157]:
predictions = np.array([])

for x in ds_test_batch:
  text_item = x['text'] if isinstance(x['text'], str) else x['text'][0]
  label_item = x['label'] if isinstance(x['label'], str) else x['label'][0]
  tokenized = tokenizer(text_item, padding="max_length", truncation=True, return_tensors="pt")
  model.eval()
  
  with torch.no_grad():
    outputs = model(**tokenized)
    pred = torch.argmax(outputs.logits, dim=-1)
    predictions = np.append(predictions, pred.item())

print(predictions)

[0. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0.
 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 1. 0. 0. 0.
 0. 0.]


In [ ]:
ds_reshaped = ds_test.reshape(10, -1)[0]

[1 1 0 1 0 1 1 0 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 1 0 1 0 1 1 1 0 0 1 0 0 1 0
 0 0 0 0 0 0 0 0 1 1 1 1 0]


In [ ]:

test_accuracy(predictions, ds_reshaped)
test_precision(predictions, ds_reshaped)
test_recall(predictions, ds_reshaped)
test_f1score(predictions.reshape(10, -1), ds_reshaped.reshape(10, -1))

In [159]:
print("Test set accuracy: {:.2%}".format(test_accuracy.result()))
print("Test set precision: {:.2%}".format(test_precision.result()))
print("Test set recall: {:.2%}".format(test_recall.result()))
print("Test set f1 score: {:.2}".format(test_f1score.result()))

Test set accuracy: 52.00%
Test set precision: 50.00%
Test set recall: 45.83%
Test set f1 score: 0.47


In [160]:
import pandas as pd
from sklearn.metrics import confusion_matrix

labels = [1,0]
confmatrix =  confusion_matrix(ds_reshaped, predictions, labels=labels)
pd.DataFrame(confmatrix, index=['POS', 'NEG'], columns=['POS', 'NEG'])

,POS,NEG
POS,11,11
NEG,13,15


In [178]:
false = [i for i, x in enumerate(ds_reshaped) if predictions[i] != x]
print(false)
false_neg = [i for i in false if ds_reshaped[i] == 0]
print(false_neg)

[0, 2, 4, 6, 7, 10, 11, 14, 15, 17, 23, 24, 26, 30, 32, 34, 37, 39, 41, 44, 45, 46, 47, 48]
[2, 4, 7, 11, 17, 24, 26, 30, 34, 37, 39, 41, 44]


In [186]:
for x, i in enumerate(false_neg[:2]):
    print(f"IDX {i}. {small_test_dataset[x]['text']}")

print("CASE 1 (IDX 2): This misclassification was likely due to the many negative words (abuse, angry, denial) used in the review, creating ambiguity.")
print("CASE 2 (IDX 4): This misclassification was likely due to the reviewer pointing out problematic aspects of the film, creating ambiguity.")


IDX 2. <br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it. I truly hope A Thousand Acres has had the same effect on some others out there.<br /><br /